# Causal-STGCN QTraffic Phase 2

Phase-2 notebook for the gate + external bypass setting. This version includes:

1. Explicit `channel_schema`
2. Query as a node-local branch
3. Weather as a global branch
4. Past-only rolling dynamic graphs
5. Topology overlap diagnostics and weather/query ablations


In [ ]:
# ===== 0. Notebook environment =====
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data" / "qtraffic").exists() and (PROJECT_ROOT.parent / "data" / "qtraffic").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT / "code") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "code"))

print("PROJECT_ROOT =", PROJECT_ROOT)
print("CODE_DIR =", PROJECT_ROOT / "code")
print("QTRAFFIC_DIR =", PROJECT_ROOT / "data" / "qtraffic")


In [ ]:
# ===== 1. Config =====
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Tuple


def _default_channel_schema() -> Dict[str, Dict[str, object]]:
    return {
        "traffic_only": {
            "feature_names": ["speed"],
            "main_channels": 1,
            "local_ext_channels": 0,
            "global_ext_channels": 0,
        },
        "traffic_time": {
            "feature_names": ["speed", "time_sin", "time_cos", "day_sin", "day_cos"],
            "main_channels": 5,
            "local_ext_channels": 0,
            "global_ext_channels": 0,
        },
        "traffic_time_weather": {
            "feature_names": [
                "speed",
                "time_sin",
                "time_cos",
                "day_sin",
                "day_cos",
                "temperature",
                "wind_speed",
            ],
            "main_channels": 5,
            "local_ext_channels": 0,
            "global_ext_channels": 2,
        },
        "traffic_time_query": {
            "feature_names": ["speed", "time_sin", "time_cos", "day_sin", "day_cos", "query"],
            "main_channels": 5,
            "local_ext_channels": 1,
            "global_ext_channels": 0,
        },
        "traffic_time_weather_query": {
            "feature_names": [
                "speed",
                "time_sin",
                "time_cos",
                "day_sin",
                "day_cos",
                "query",
                "temperature",
                "wind_speed",
            ],
            "main_channels": 5,
            "local_ext_channels": 1,
            "global_ext_channels": 2,
        },
    }


@dataclass
class ExperimentConfig:
    qtraffic_dir: str = "./data/qtraffic"
    project_root: Path = field(default_factory=lambda: Path.cwd().resolve())

    seq_len: int = 12
    horizons: Tuple[int, ...] = (12, 36)
    report_steps: Tuple[int, ...] = (12, 36)
    target_feature_idx: int = 0
    slot_minutes: int = 15

    train_ratio: float = 0.7
    val_ratio: float = 0.1
    batch_size: int = 64

    lr: float = 1e-3
    weight_decay: float = 1e-4
    epochs: int = 80
    grad_clip: float = 5.0
    early_stop_patience: int = 12

    epoch_ckpt_enabled: bool = True
    epoch_ckpt_save_every: int = 1
    epoch_ckpt_restore_rng: bool = False
    epoch_ckpt_keep_finished: bool = False

    hidden: int = 64
    blocks: int = 2
    kt: int = 3
    dilation_base: int = 2
    dropout: float = 0.3
    diffusion_K: int = 2
    use_backward_diffusion: bool = True

    y_null_val: float = 0.0
    mape_min_denom: float = 1.0

    lambdas_full: Tuple[float, ...] = (0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0)
    lambdas_fast: Tuple[float, ...] = (0.0, 0.5, 1.0)

    dynamic_graph_mode: str = "granger_rolling"
    dynamic_graph_eval_mode: str = "past_only"
    graph_snapshot_stride: int = 96
    graph_history_window: int = 96 * 7
    granger_max_lag: int = 3
    granger_neighbor_topk: int = 15
    granger_f_threshold: float = 1.5
    granger_min_seg_len: int = 64
    granger_cache_enabled: bool = True

    run_baseline_ha: bool = True
    run_baseline_arima: bool = True
    run_baseline_dcrnn: bool = True
    dcrnn_use_dyn: bool = False
    dcrnn_hidden: int = 32
    dcrnn_epochs: int = 40
    dcrnn_patience: int = 8

    arima_p: int = 3

    channel_schema: Dict[str, Dict[str, object]] = field(default_factory=_default_channel_schema)
    result_dir: str = "./results/qtraffic_phase2_gate_bypass_fix_v1"


In [ ]:
# ===== 2. Data =====
import json
import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import torch

# ExperimentConfig is already defined in a previous notebook cell.


def set_seed(seed: int) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def load_first_existing(candidates):
    for path in candidates:
        if path.exists():
            return np.load(path).astype(np.float32), path.name
    raise FileNotFoundError("No candidate file found: " + ", ".join(str(p) for p in candidates))


def resolve_cfg_path(path_like, project_root: Path) -> Path:
    path = Path(path_like)
    if path.exists():
        return path
    if not path.is_absolute():
        alt = (project_root / path).resolve()
        if alt.exists():
            return alt
        return alt
    return path


def _ols_rss(X, y) -> float:
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    return float(np.dot(resid, resid))


def granger_f_score(x, y, max_lag=3) -> float:
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    total = len(y)
    best_f = 0.0

    for lag in range(1, int(max_lag) + 1):
        if total <= (4 * lag + 8):
            continue

        target = y[lag:]
        y_lags = [y[lag - k : total - k] for k in range(1, lag + 1)]
        x_lags = [x[lag - k : total - k] for k in range(1, lag + 1)]

        X_r = np.column_stack(y_lags + [np.ones(len(target))])
        X_f = np.column_stack(y_lags + x_lags + [np.ones(len(target))])

        rss_r = _ols_rss(X_r, target)
        rss_f = _ols_rss(X_f, target)

        df1 = lag
        df2 = max(1, len(target) - (2 * lag + 1))

        if rss_f <= 1e-12:
            f_val = 0.0
        else:
            f_val = ((rss_r - rss_f) / max(1e-12, df1)) / (rss_f / max(1e-12, df2))
            if not np.isfinite(f_val):
                f_val = 0.0
            f_val = float(max(0.0, f_val))

        if f_val > best_f:
            best_f = f_val

    return float(best_f)


def _select_granger_candidates(A_phy: np.ndarray, target_idx: int, neighbor_topk: int) -> np.ndarray:
    A_bin = A_phy > 0
    candidates = np.where(A_bin[target_idx] | A_bin[:, target_idx])[0]
    candidates = candidates[candidates != target_idx]

    if len(candidates) == 0:
        row = A_phy[target_idx].copy()
        row[target_idx] = 0.0
        candidates = np.argsort(-row)[: int(neighbor_topk)]

    if len(candidates) > int(neighbor_topk):
        candidates = candidates[: int(neighbor_topk)]

    return candidates


def build_single_granger_graph(
    speed_window: np.ndarray,
    A_phy: np.ndarray,
    max_lag: int,
    neighbor_topk: int,
    f_threshold: float,
) -> np.ndarray:
    speed = np.asarray(speed_window, dtype=np.float32)
    if speed.ndim == 3:
        speed = speed[:, :, 0]

    _, num_nodes = speed.shape
    A_dyn = np.zeros((num_nodes, num_nodes), dtype=np.float32)

    for tgt in range(num_nodes):
        candidates = _select_granger_candidates(A_phy, tgt, neighbor_topk)
        y = speed[:, tgt]
        for src in candidates:
            x = speed[:, src]
            f_val = granger_f_score(x, y, max_lag=max_lag)
            if f_val >= float(f_threshold):
                A_dyn[src, tgt] = float(f_val)

    mx = float(A_dyn.max())
    if mx > 1e-12:
        A_dyn /= mx
    np.fill_diagonal(A_dyn, 1.0)
    return A_dyn.astype(np.float32)


def _build_snapshot_end_times(total_steps: int, seq_len: int, snapshot_stride: int) -> np.ndarray:
    start = int(max(0, seq_len - 1))
    stride = int(max(1, snapshot_stride))
    offsets = [start]
    offsets.extend(range(stride - 1, total_steps, stride))
    offsets = sorted(set(int(x) for x in offsets if 0 <= int(x) < total_steps))
    return np.asarray(offsets, dtype=np.int64)


def build_granger_rolling_snapshots(
    speed_only: np.ndarray,
    A_phy: np.ndarray,
    seq_len: int,
    snapshot_stride: int,
    history_window: int,
    max_lag: int,
    neighbor_topk: int,
    f_threshold: float,
    min_seg_len: int,
) -> Tuple[np.ndarray, np.ndarray]:
    speed = np.asarray(speed_only, dtype=np.float32)
    if speed.ndim == 3:
        speed = speed[:, :, 0]

    total_steps = int(speed.shape[0])
    snapshot_end_times = _build_snapshot_end_times(total_steps, seq_len, snapshot_stride)
    graphs = np.zeros((len(snapshot_end_times), speed.shape[1], speed.shape[1]), dtype=np.float32)

    for idx, end_t in enumerate(snapshot_end_times):
        start_t = max(0, int(end_t) + 1 - int(max(1, history_window)))
        speed_window = speed[start_t : int(end_t) + 1]

        if speed_window.shape[0] < int(max(1, min_seg_len)):
            graphs[idx] = np.eye(speed.shape[1], dtype=np.float32)
            continue

        graphs[idx] = build_single_granger_graph(
            speed_window=speed_window,
            A_phy=A_phy,
            max_lag=max_lag,
            neighbor_topk=neighbor_topk,
            f_threshold=f_threshold,
        )

    return graphs.astype(np.float32), snapshot_end_times.astype(np.int64)


def _rolling_cache_stem(cfg: ExperimentConfig, num_nodes: int) -> str:
    return (
        f"A_dynamic_granger_roll_{int(num_nodes)}nodes_"
        f"lag{int(cfg.granger_max_lag)}_"
        f"k{int(cfg.granger_neighbor_topk)}_"
        f"f{str(cfg.granger_f_threshold).replace('.', 'p')}_"
        f"stride{int(cfg.graph_snapshot_stride)}_"
        f"hist{int(cfg.graph_history_window)}"
    )


def build_or_load_rolling_dynamic_graph(
    cfg: ExperimentConfig,
    data_dir: Path,
    speed_only: np.ndarray,
    A_phy: np.ndarray,
) -> Tuple[np.ndarray, np.ndarray, str]:
    stem = _rolling_cache_stem(cfg, A_phy.shape[0])
    cache_npz = data_dir / f"{stem}.npz"
    cache_meta = data_dir / f"{stem}.json"

    if bool(cfg.granger_cache_enabled) and cache_npz.exists():
        pack = np.load(cache_npz)
        A_dyn = pack["A_dynamic_tensor"].astype(np.float32)
        snapshot_end_times = pack["snapshot_end_times"].astype(np.int64)
        return A_dyn, snapshot_end_times, f"granger_rolling_cached:{cache_npz.name}"

    A_dyn, snapshot_end_times = build_granger_rolling_snapshots(
        speed_only=speed_only,
        A_phy=A_phy,
        seq_len=cfg.seq_len,
        snapshot_stride=cfg.graph_snapshot_stride,
        history_window=cfg.graph_history_window,
        max_lag=cfg.granger_max_lag,
        neighbor_topk=cfg.granger_neighbor_topk,
        f_threshold=cfg.granger_f_threshold,
        min_seg_len=cfg.granger_min_seg_len,
    )

    if bool(cfg.granger_cache_enabled):
        np.savez_compressed(
            cache_npz,
            A_dynamic_tensor=A_dyn.astype(np.float32),
            snapshot_end_times=snapshot_end_times.astype(np.int64),
        )
        cache_payload = {
            "graph_window": int(cfg.graph_history_window),
            "snapshot_stride": int(cfg.graph_snapshot_stride),
            "graph_source_range": [0, int(speed_only.shape[0]) - 1],
            "snapshot_count": int(len(snapshot_end_times)),
            "snapshot_end_times_preview": snapshot_end_times[: min(10, len(snapshot_end_times))].tolist(),
        }
        with open(cache_meta, "w", encoding="utf-8") as f:
            json.dump(cache_payload, f, ensure_ascii=False, indent=2)

    return A_dyn, snapshot_end_times, f"granger_rolling_built:{cache_npz.name}"


def infer_channel_names(meta: Dict) -> Dict[str, Sequence[str]]:
    stage2 = meta.get("stage2", {}) if isinstance(meta.get("stage2", {}), dict) else {}
    weather_names = list(stage2.get("weather_kept_feature_names", ["temperature", "wind_speed"]))
    if len(weather_names) == 0:
        weather_names = ["temperature", "wind_speed"]
    return {
        "time": ["time_sin", "time_cos", "day_sin", "day_cos"],
        "weather": weather_names,
        "query": ["query"],
    }


def build_feature_views(
    traffic_only: np.ndarray,
    traffic_time_raw: np.ndarray,
    traffic_time_weather_raw: np.ndarray,
    traffic_time_weather_query_raw: np.ndarray,
    query_feature: np.ndarray,
    meta: Dict,
) -> Tuple[Dict[str, np.ndarray], Dict[str, Dict[str, object]]]:
    names = infer_channel_names(meta)
    weather_dim = int(traffic_time_weather_raw.shape[-1] - 5)
    weather_names = list(names["weather"][:weather_dim])
    if len(weather_names) < weather_dim:
        weather_names.extend([f"weather_{i}" for i in range(len(weather_names), weather_dim)])

    traffic_time = traffic_time_raw[:, :, :5].astype(np.float32)
    traffic_time_weather = traffic_time_weather_raw[:, :, : 5 + weather_dim].astype(np.float32)
    traffic_time_query = np.concatenate([traffic_time, query_feature], axis=-1).astype(np.float32)

    if traffic_time_weather_query_raw.shape[-1] == 5 + weather_dim + 1:
        traffic_time_weather_query = np.concatenate(
            [
                traffic_time_weather_query_raw[:, :, :5],
                traffic_time_weather_query_raw[:, :, -1:],
                traffic_time_weather_query_raw[:, :, 5 : 5 + weather_dim],
            ],
            axis=-1,
        ).astype(np.float32)
    else:
        traffic_time_weather_query = np.concatenate(
            [traffic_time, query_feature, traffic_time_weather[:, :, 5:]],
            axis=-1,
        ).astype(np.float32)

    views = {
        "traffic_only": traffic_only.astype(np.float32),
        "traffic_time": traffic_time.astype(np.float32),
        "traffic_time_weather": traffic_time_weather.astype(np.float32),
        "traffic_time_query": traffic_time_query.astype(np.float32),
        "traffic_time_weather_query": traffic_time_weather_query.astype(np.float32),
    }

    channel_schema = {
        "traffic_only": {
            "feature_names": ["speed"],
            "main_channels": 1,
            "local_ext_channels": 0,
            "global_ext_channels": 0,
        },
        "traffic_time": {
            "feature_names": ["speed", *names["time"]],
            "main_channels": 5,
            "local_ext_channels": 0,
            "global_ext_channels": 0,
        },
        "traffic_time_weather": {
            "feature_names": ["speed", *names["time"], *weather_names],
            "main_channels": 5,
            "local_ext_channels": 0,
            "global_ext_channels": weather_dim,
        },
        "traffic_time_query": {
            "feature_names": ["speed", *names["time"], *names["query"]],
            "main_channels": 5,
            "local_ext_channels": 1,
            "global_ext_channels": 0,
        },
        "traffic_time_weather_query": {
            "feature_names": ["speed", *names["time"], *names["query"], *weather_names],
            "main_channels": 5,
            "local_ext_channels": 1,
            "global_ext_channels": weather_dim,
        },
    }
    return views, channel_schema


def compute_topology_overlap_proxy(A_dyn: np.ndarray, A_phy: np.ndarray) -> Dict[str, float]:
    score = np.asarray(A_dyn, dtype=np.float64)
    if score.ndim == 3:
        score_mean = score.mean(axis=0)
    else:
        score_mean = score

    A_phy = np.asarray(A_phy, dtype=np.float64)
    mask = ~np.eye(A_phy.shape[0], dtype=bool)

    phy_edges = (A_phy > 0)[mask]
    dyn_edges = (score_mean > 0)[mask]

    intersection = np.logical_and(phy_edges, dyn_edges).sum()
    union = np.logical_or(phy_edges, dyn_edges).sum()
    phy_total = max(1, int(phy_edges.sum()))
    dyn_total = max(1, int(dyn_edges.sum()))

    persistence = float((score[:, mask] > 0).mean()) if score.ndim == 3 else float(dyn_edges.mean())
    temporal_variation = float(score[:, mask].std(axis=0).mean()) if score.ndim == 3 else 0.0

    return {
        "edge_jaccard": float(intersection / max(1, union)),
        "physical_recall": float(intersection / phy_total),
        "dynamic_precision": float(intersection / dyn_total),
        "edge_persistence": persistence,
        "temporal_variation": temporal_variation,
        "num_physical_edges": float(phy_edges.sum()),
        "num_dynamic_edges": float(dyn_edges.sum()),
    }


@dataclass
class QTrafficDataBundle:
    meta: Dict
    adj_matrix: np.ndarray
    A_dynamic_proxy: np.ndarray
    A_dynamic_tensor: np.ndarray
    snapshot_end_times: np.ndarray
    feature_views: Dict[str, np.ndarray]
    channel_schema: Dict[str, Dict[str, object]]
    slots_per_day: int
    slot_minutes: int
    dynamic_graph_source: str
    speed_source_name: str
    time_source_name: str
    time_weather_source_name: str
    full_source_name: str
    topology_overlap_proxy: Dict[str, float]

    @property
    def traffic_only(self) -> np.ndarray:
        return self.feature_views["traffic_only"]

    @property
    def traffic_time(self) -> np.ndarray:
        return self.feature_views["traffic_time"]

    @property
    def traffic_time_weather(self) -> np.ndarray:
        return self.feature_views["traffic_time_weather"]

    @property
    def traffic_time_query(self) -> np.ndarray:
        return self.feature_views["traffic_time_query"]

    @property
    def traffic_time_weather_query(self) -> np.ndarray:
        return self.feature_views["traffic_time_weather_query"]


def load_qtraffic_bundle(cfg: ExperimentConfig) -> QTrafficDataBundle:
    data_dir = resolve_cfg_path(cfg.qtraffic_dir, cfg.project_root)
    meta = {}
    meta_file = data_dir / "qtraffic_metadata.json"
    if meta_file.exists():
        with open(meta_file, "r", encoding="utf-8") as f:
            meta = json.load(f)

    adj_matrix = np.load(data_dir / "adj_mat_500nodes.npy").astype(np.float32)
    A_dynamic_proxy = np.load(data_dir / "A_dynamic_tensor_500nodes.npy").astype(np.float32)

    speed_candidates = [
        data_dir / "node_values_500nodes.npy",
        data_dir / "X_fused_500nodes.npy",
    ]
    speed_tensor, speed_source_name = load_first_existing(speed_candidates)
    traffic_only = speed_tensor[:, :, 0:1].astype(np.float32)

    time_candidates = [
        data_dir / "X_fused_timeonly_5dim.npy",
        data_dir / "X_fused_500nodes.npy",
    ]
    traffic_time_raw, time_source_name = load_first_existing(time_candidates)

    time_weather_candidates = [
        data_dir / "X_fused_time_weather_7dim.npy",
        data_dir / "X_fused_time_weather.npy",
        data_dir / "X_fused_full_7dim.npy",
        data_dir / "X_fused_full.npy",
    ]
    traffic_time_weather_raw, time_weather_source_name = load_first_existing(time_weather_candidates)

    full_candidates = [
        data_dir / "X_fused_full_query_8dim.npy",
        data_dir / "X_fused_full_query.npy",
        data_dir / "X_fused_500nodes.npy",
    ]
    traffic_time_weather_query_raw, full_source_name = load_first_existing(full_candidates)

    query_feature = np.load(data_dir / "query_features_500nodes.npy").astype(np.float32)

    feature_views, channel_schema = build_feature_views(
        traffic_only=traffic_only,
        traffic_time_raw=traffic_time_raw,
        traffic_time_weather_raw=traffic_time_weather_raw,
        traffic_time_weather_query_raw=traffic_time_weather_query_raw,
        query_feature=query_feature,
        meta=meta,
    )

    total_steps = int(feature_views["traffic_only"].shape[0])
    slots_per_day = int(meta.get("slots_per_day", 96 if total_steps % 96 == 0 else 288))
    cfg.slot_minutes = int(meta.get("slot_minutes", 1440 // slots_per_day))

    A_dynamic_tensor, snapshot_end_times, dynamic_graph_source = build_or_load_rolling_dynamic_graph(
        cfg=cfg,
        data_dir=data_dir,
        speed_only=traffic_only,
        A_phy=adj_matrix,
    )
    topology_overlap_proxy = compute_topology_overlap_proxy(A_dynamic_tensor, adj_matrix)

    cfg.channel_schema = channel_schema

    return QTrafficDataBundle(
        meta=meta,
        adj_matrix=adj_matrix,
        A_dynamic_proxy=A_dynamic_proxy,
        A_dynamic_tensor=A_dynamic_tensor,
        snapshot_end_times=snapshot_end_times,
        feature_views=feature_views,
        channel_schema=channel_schema,
        slots_per_day=slots_per_day,
        slot_minutes=cfg.slot_minutes,
        dynamic_graph_source=dynamic_graph_source,
        speed_source_name=speed_source_name,
        time_source_name=time_source_name,
        time_weather_source_name=time_weather_source_name,
        full_source_name=full_source_name,
        topology_overlap_proxy=topology_overlap_proxy,
    )


class FeatureScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, x):
        x = np.asarray(x, dtype=np.float32)
        self.mean = x.mean(axis=(0, 1, 2), keepdims=True).astype(np.float32)
        self.std = x.std(axis=(0, 1, 2), keepdims=True).astype(np.float32)
        self.std[self.std == 0] = 1.0
        return self

    def transform(self, x):
        x = np.asarray(x, dtype=np.float32)
        return ((x - self.mean) / self.std).astype(np.float32)


class TargetScaler:
    def __init__(self):
        self.mean = 0.0
        self.std = 1.0

    def fit(self, y_raw, mask_raw):
        y = np.asarray(y_raw, dtype=np.float64).reshape(-1)
        m = np.asarray(mask_raw).reshape(-1).astype(bool)
        vals = y[m]
        self.mean = float(vals.mean())
        self.std = float(vals.std() + 1e-6)
        return self

    def transform(self, y_raw):
        y = np.asarray(y_raw, dtype=np.float64)
        return ((y - self.mean) / self.std).astype(np.float32)

    def inverse(self, y_norm):
        y = np.asarray(y_norm, dtype=np.float64)
        return (y * self.std + self.mean).astype(np.float32)


def masked_metrics_np(y_true_raw, y_pred_raw, mask_raw, min_denom=1.0):
    yt = np.asarray(y_true_raw).reshape(-1)
    yp = np.asarray(y_pred_raw).reshape(-1)
    mask = np.asarray(mask_raw).reshape(-1).astype(bool)

    keep = float(mask.mean())
    if keep < 1e-6:
        return {"MAE": float("nan"), "RMSE": float("nan"), "MAPE": float("nan"), "keep%": keep * 100}

    yt_masked = yt[mask]
    yp_masked = yp[mask]
    mae = float(np.mean(np.abs(yp_masked - yt_masked)))
    rmse = float(np.sqrt(np.mean((yp_masked - yt_masked) ** 2)))

    denom_mask = np.abs(yt_masked) >= float(min_denom)
    if denom_mask.mean() < 1e-6:
        mape = float("nan")
    else:
        yt2 = yt_masked[denom_mask]
        yp2 = yp_masked[denom_mask]
        mape = float(np.mean(np.abs((yp2 - yt2) / yt2)))

    return {"MAE": mae, "RMSE": rmse, "MAPE": mape, "keep%": keep * 100}


def build_samples(X_used, seq_len, H, target_idx):
    num_samples = X_used.shape[0] - seq_len - H + 1
    if num_samples <= 0:
        return (
            np.empty((0, seq_len, X_used.shape[1], X_used.shape[-1]), dtype=np.float32),
            np.empty((0, H, X_used.shape[1]), dtype=np.float32),
        )
    X_all = np.empty((num_samples, seq_len, X_used.shape[1], X_used.shape[-1]), dtype=np.float32)
    y_raw = np.empty((num_samples, H, X_used.shape[1]), dtype=np.float32)
    for i in range(num_samples):
        X_all[i] = X_used[i : i + seq_len]
        y_raw[i] = X_used[i + seq_len : i + seq_len + H, :, target_idx]
    return X_all, y_raw


def split_timeline(X_mode, cfg: ExperimentConfig):
    total_steps = X_mode.shape[0]
    train_steps = int(total_steps * cfg.train_ratio)
    val_steps = int(total_steps * (cfg.train_ratio + cfg.val_ratio))
    return X_mode[:train_steps], X_mode[train_steps:val_steps], X_mode[val_steps:], train_steps, val_steps


def build_graph_indices_from_snapshots(
    num_samples: int,
    seq_len: int,
    snapshot_end_times: np.ndarray,
    start_t: int = 0,
) -> np.ndarray:
    if num_samples <= 0:
        return np.empty((0,), dtype=np.int64)

    snapshot_end_times = np.asarray(snapshot_end_times, dtype=np.int64)
    idx = np.empty((num_samples,), dtype=np.int64)

    for sample_idx in range(num_samples):
        input_window_end = int(start_t) + int(sample_idx) + int(seq_len) - 1
        g = int(np.searchsorted(snapshot_end_times, input_window_end, side="right") - 1)
        idx[sample_idx] = max(0, g)

    return idx


In [ ]:
# ===== 3. Models =====
import math
from typing import Dict, Optional, Tuple

import torch
import torch.nn as nn
import torch.nn.functional as F


def normalize_adj_rw(A_batch: torch.Tensor, eps: float = 1e-6, add_self_loop: bool = True) -> torch.Tensor:
    if A_batch.dim() == 2:
        A_batch = A_batch.unsqueeze(0)
    batch, num_nodes, _ = A_batch.shape

    diag_mask = torch.eye(num_nodes, device=A_batch.device, dtype=torch.bool).unsqueeze(0).expand(batch, -1, -1)
    A_batch = A_batch.masked_fill(diag_mask, 0.0)

    if add_self_loop:
        eye = torch.eye(num_nodes, device=A_batch.device).unsqueeze(0).expand(batch, -1, -1)
        A_batch = A_batch + eye

    A_batch = torch.clamp(A_batch, min=0.0)
    degree = A_batch.sum(dim=-1).clamp(min=eps)
    return A_batch / degree.unsqueeze(-1)


class NodeTimeLayerNorm(nn.Module):
    def __init__(self, channels: int, eps: float = 1e-5, affine: bool = True):
        super().__init__()
        self.ln = nn.LayerNorm(channels, eps=eps, elementwise_affine=affine)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 3, 1).contiguous()
        x = self.ln(x)
        return x.permute(0, 3, 1, 2).contiguous()


class CausalConv2d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, dilation: int = 1, bias: bool = True):
        super().__init__()
        self.left_pad = int(max(0, (kernel_size - 1) * dilation))
        self.conv = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=(1, kernel_size),
            dilation=(1, dilation),
            bias=bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.pad(x, (self.left_pad, 0, 0, 0))
        return self.conv(x)


class CausalConv1d(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, dilation: int = 1, bias: bool = True):
        super().__init__()
        self.left_pad = int(max(0, (kernel_size - 1) * dilation))
        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            dilation=dilation,
            bias=bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.pad(x, (self.left_pad, 0))
        return self.conv(x)


class TemporalGLU(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, kernel_size: int = 3, dilation: int = 1, dropout: float = 0.0):
        super().__init__()
        self.filter = CausalConv2d(in_channels, out_channels, kernel_size, dilation)
        self.gate = CausalConv2d(in_channels, out_channels, kernel_size, dilation)
        self.residual = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1)) if in_channels != out_channels else None
        self.dropout = nn.Dropout(dropout)
        self.norm = NodeTimeLayerNorm(out_channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = torch.tanh(self.filter(x)) * torch.sigmoid(self.gate(x))
        if self.residual is not None:
            h = h + self.residual(x)
        h = self.dropout(h)
        return self.norm(h)


class DiffusionGraphConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, K: int = 2, use_backward: bool = True, dropout: float = 0.0):
        super().__init__()
        self.K = int(K)
        self.use_backward = bool(use_backward)
        num_supports = 1 + (1 if self.use_backward else 0)
        total_in = in_channels * (1 + num_supports * self.K)
        self.mlp = nn.Conv2d(total_in, out_channels, kernel_size=(1, 1))
        self.dropout = nn.Dropout(dropout)
        self.norm = NodeTimeLayerNorm(out_channels)

    def forward(self, x: torch.Tensor, A_batch: torch.Tensor) -> torch.Tensor:
        A_fw = normalize_adj_rw(A_batch)
        supports = [A_fw]
        if self.use_backward:
            supports.append(normalize_adj_rw(A_batch.transpose(1, 2)))

        outs = [x]
        for support in supports:
            xk = x
            for _ in range(self.K):
                xk = torch.einsum("bnm,bcmt->bcnt", support, xk)
                outs.append(xk)

        h = torch.cat(outs, dim=1)
        h = self.mlp(h)
        h = F.relu(h)
        h = self.dropout(h)
        return self.norm(h)


class AdaptiveDualGraphConv(nn.Module):
    def __init__(
        self,
        channels: int,
        K: int = 2,
        use_backward: bool = True,
        dropout: float = 0.0,
        gate_context_channels: int = 0,
    ):
        super().__init__()
        self.g_phy = DiffusionGraphConv(channels, channels, K=K, use_backward=use_backward, dropout=dropout)
        self.g_dyn = DiffusionGraphConv(channels, channels, K=K, use_backward=use_backward, dropout=dropout)
        gate_hidden = max(8, channels // 2)
        gate_in = int(channels + max(0, gate_context_channels))
        self.gate_net = nn.Sequential(
            nn.Conv2d(gate_in, gate_hidden, kernel_size=(1, 1)),
            nn.ReLU(),
            nn.Conv2d(gate_hidden, 1, kernel_size=(1, 1)),
        )

    def forward(
        self,
        x: torch.Tensor,
        A_phy_batch: torch.Tensor,
        A_dyn_batch: Optional[torch.Tensor] = None,
        lam_prior: float = 0.5,
        gate_context: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        h_phy = self.g_phy(x, A_phy_batch)
        if A_dyn_batch is None:
            return h_phy, None

        h_dyn = self.g_dyn(x, A_dyn_batch)
        gate_input = x.mean(dim=-1, keepdim=True)
        if gate_context is not None:
            gate_input = torch.cat([gate_input, gate_context], dim=1)

        lam_prior = float(max(1e-4, min(1.0 - 1e-4, lam_prior)))
        prior_logit = math.log(lam_prior / (1.0 - lam_prior))
        gate = torch.sigmoid(self.gate_net(gate_input) + prior_logit)
        h = (1.0 - gate) * h_phy + gate * h_dyn
        return h, gate


class STConvBlock(nn.Module):
    def __init__(
        self,
        channels: int,
        kt: int = 3,
        dilation: int = 1,
        dropout: float = 0.0,
        K: int = 2,
        use_backward: bool = True,
        gate_context_channels: int = 0,
    ):
        super().__init__()
        self.t1 = TemporalGLU(channels, channels, kernel_size=kt, dilation=dilation, dropout=dropout)
        self.g = AdaptiveDualGraphConv(
            channels,
            K=K,
            use_backward=use_backward,
            dropout=dropout,
            gate_context_channels=gate_context_channels,
        )
        self.t2 = TemporalGLU(channels, channels, kernel_size=kt, dilation=dilation, dropout=dropout)

    def forward(
        self,
        x: torch.Tensor,
        A_phy_batch: torch.Tensor,
        A_dyn_batch: Optional[torch.Tensor] = None,
        lam: float = 0.5,
        gate_context: Optional[torch.Tensor] = None,
    ) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        x = self.t1(x)
        x, gate = self.g(x, A_phy_batch, A_dyn_batch, lam_prior=lam, gate_context=gate_context)
        x = self.t2(x)
        return x, gate


class NodeLocalExternalHead(nn.Module):
    def __init__(self, in_channels: int, hidden: int, kernel_size: int = 3, dropout: float = 0.0):
        super().__init__()
        self.proj = nn.Conv2d(in_channels, hidden, kernel_size=(1, 1))
        self.temporal = TemporalGLU(hidden, hidden, kernel_size=kernel_size, dilation=1, dropout=dropout)

    def forward(self, x: Optional[torch.Tensor]) -> Tuple[Optional[torch.Tensor], Optional[torch.Tensor]]:
        if x is None or x.shape[1] == 0:
            return None, None
        h = self.temporal(self.proj(x))
        node_feat = h[:, :, :, -1].permute(0, 2, 1).contiguous()
        gate_context = h.mean(dim=-1, keepdim=True)
        return node_feat, gate_context


class GlobalExternalHead(nn.Module):
    def __init__(self, in_channels: int, hidden: int, kernel_size: int = 3, dropout: float = 0.0):
        super().__init__()
        self.proj = nn.Conv1d(in_channels, hidden, kernel_size=1)
        self.filter = CausalConv1d(hidden, hidden, kernel_size=kernel_size, dilation=1)
        self.gate = CausalConv1d(hidden, hidden, kernel_size=kernel_size, dilation=1)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: Optional[torch.Tensor], num_nodes: int) -> Optional[torch.Tensor]:
        if x is None or x.shape[1] == 0:
            return None
        pooled = x.mean(dim=2)
        pooled = self.proj(pooled)
        h = torch.tanh(self.filter(pooled)) * torch.sigmoid(self.gate(pooled))
        h = self.dropout(h)
        h_last = h[:, :, -1]
        return h_last.unsqueeze(1).expand(-1, num_nodes, -1)


class CausalSTGCN(nn.Module):
    def __init__(
        self,
        in_channels: int,
        main_channels: int,
        local_ext_channels: int = 0,
        global_ext_channels: int = 0,
        hidden: int = 64,
        blocks: int = 2,
        kt: int = 3,
        dilation_base: int = 2,
        dropout: float = 0.3,
        K: int = 2,
        use_backward: bool = True,
        H: int = 12,
    ):
        super().__init__()
        self.main_channels = int(main_channels)
        self.local_ext_channels = int(local_ext_channels)
        self.global_ext_channels = int(global_ext_channels)
        total_channels = self.main_channels + self.local_ext_channels + self.global_ext_channels
        if total_channels != int(in_channels):
            raise ValueError(
                f"channel split mismatch: main/local/global={self.main_channels}/"
                f"{self.local_ext_channels}/{self.global_ext_channels}, in_channels={in_channels}"
            )

        self.in_proj = nn.Conv2d(self.main_channels, hidden, kernel_size=(1, 1))
        gate_context_channels = hidden if self.local_ext_channels > 0 else 0
        self.blocks = nn.ModuleList(
            [
                STConvBlock(
                    hidden,
                    kt=kt,
                    dilation=(dilation_base ** i),
                    dropout=dropout,
                    K=K,
                    use_backward=use_backward,
                    gate_context_channels=gate_context_channels,
                )
                for i in range(blocks)
            ]
        )

        self.local_head = (
            NodeLocalExternalHead(self.local_ext_channels, hidden, kernel_size=kt, dropout=dropout)
            if self.local_ext_channels > 0
            else None
        )
        self.global_head = (
            GlobalExternalHead(self.global_ext_channels, hidden, kernel_size=kt, dropout=dropout)
            if self.global_ext_channels > 0
            else None
        )

        readout_dim = hidden
        if self.local_head is not None:
            readout_dim += hidden
        if self.global_head is not None:
            readout_dim += hidden
        self.readout = nn.Linear(readout_dim, H)

    def forward(
        self,
        x: torch.Tensor,
        A_phy_batch: torch.Tensor,
        A_dyn_batch: Optional[torch.Tensor] = None,
        lam: float = 0.0,
    ) -> Tuple[torch.Tensor, Dict[str, Optional[torch.Tensor]]]:
        start = 0
        x_main = x[:, start : start + self.main_channels]
        start += self.main_channels
        x_local = x[:, start : start + self.local_ext_channels] if self.local_ext_channels > 0 else None
        start += self.local_ext_channels
        x_global = x[:, start : start + self.global_ext_channels] if self.global_ext_channels > 0 else None

        h = self.in_proj(x_main)
        local_node_feat, gate_context = self.local_head(x_local) if self.local_head is not None else (None, None)
        global_node_feat = self.global_head(x_global, h.shape[2]) if self.global_head is not None else None

        gates = []
        for block in self.blocks:
            h, gate = block(h, A_phy_batch, A_dyn_batch, lam=lam, gate_context=gate_context)
            if gate is not None:
                gates.append(gate)

        h_last = h[:, :, :, -1].permute(0, 2, 1).contiguous()
        readout_parts = [h_last]
        if local_node_feat is not None:
            readout_parts.append(local_node_feat)
        if global_node_feat is not None:
            readout_parts.append(global_node_feat)
        out = self.readout(torch.cat(readout_parts, dim=-1))

        aux = {
            "local_node_features": local_node_feat,
            "global_node_features": global_node_feat,
            "gate_context": gate_context,
            "gate_mean": None if len(gates) == 0 else torch.stack([g.mean() for g in gates]).mean(),
        }
        return out.permute(0, 2, 1).contiguous(), aux


class DCRNNBaseline(nn.Module):
    def __init__(self, in_channels: int, hidden: int = 32, H: int = 12, **kwargs):
        super().__init__()
        self.hidden = int(hidden)
        self.H = int(H)
        self.in_proj = nn.Linear(in_channels, self.hidden)
        self.gru = nn.GRU(self.hidden, self.hidden, batch_first=True)
        self.out = nn.Linear(self.hidden, self.H)

    def forward(self, x: torch.Tensor, A_batch: torch.Tensor) -> torch.Tensor:
        batch, channels, num_nodes, seq_len = x.shape
        A_rw = normalize_adj_rw(A_batch)

        x_bt_nc = x.permute(0, 3, 2, 1).contiguous()
        x_diff = torch.einsum("bnm,btmc->btnc", A_rw, x_bt_nc)

        h = self.in_proj(x_diff)
        h = h.permute(0, 2, 1, 3).contiguous().view(batch * num_nodes, seq_len, self.hidden)
        out, _ = self.gru(h)
        last = out[:, -1, :]
        y = self.out(last)
        return y.view(batch, num_nodes, self.H).permute(0, 2, 1).contiguous()


def causal_grad_test(device: Optional[torch.device] = None) -> float:
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    batch, channels, num_nodes, seq_len = 2, 4, 3, 8
    layer = TemporalGLU(channels, channels, kernel_size=3, dilation=1, dropout=0.0).to(device)
    x = torch.randn(batch, channels, num_nodes, seq_len, device=device, requires_grad=True)
    y = layer(x)
    target_t = 3
    y[:, :, :, target_t].sum().backward()
    future_grad = x.grad.detach().abs()[..., target_t + 1 :]
    if future_grad.numel() == 0:
        return 0.0
    return float(future_grad.max().item())


In [ ]:
# ===== 4. Experiment =====
import json
import random
from dataclasses import asdict, dataclass
from datetime import datetime
from math import inf
from pathlib import Path
from typing import Dict, Iterable, Optional, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset

# ExperimentConfig, data utilities, and model classes are already defined in previous cells.


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


@dataclass
class ModeSpec:
    name: str
    view_key: str
    use_dyn: bool
    model_label: str
    desc: str


class SeqDataset(Dataset):
    def __init__(self, X, y_norm, y_raw, mask_raw, a_idx=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y_norm = torch.tensor(y_norm, dtype=torch.float32)
        self.y_raw = torch.tensor(y_raw, dtype=torch.float32)
        self.mask = torch.tensor(mask_raw.astype(np.float32))
        self.a = None if a_idx is None else torch.tensor(a_idx, dtype=torch.long)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, i):
        x = self.X[i].permute(2, 1, 0).contiguous()
        yn, yr, m = self.y_norm[i], self.y_raw[i], self.mask[i]
        if self.a is None:
            return x, yn, yr, m
        return x, yn, yr, m, self.a[i]


def masked_l1_loss(pred, y_norm, mask):
    denom = torch.clamp(mask.mean(), min=1e-6)
    return torch.mean(torch.abs(pred - y_norm) * mask) / denom


def _safe_tag(text):
    return str(text).replace(" ", "_").replace("/", "_")


def run_id(seed, mode_name, H):
    seed_tag = "seedNA" if seed is None else f"seed{int(seed)}"
    return f"{seed_tag}_{_safe_tag(mode_name)}_H{int(H)}"


def save_df_atomic(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    df.to_csv(tmp, index=False)
    tmp.replace(path)


def save_json_atomic(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)
    tmp.replace(path)


def save_torch_atomic(obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    torch.save(obj, tmp)
    tmp.replace(path)


def to_cpu_obj(obj):
    if torch.is_tensor(obj):
        return obj.detach().cpu()
    if isinstance(obj, dict):
        return {k: to_cpu_obj(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [to_cpu_obj(v) for v in obj]
    if isinstance(obj, tuple):
        return tuple(to_cpu_obj(v) for v in obj)
    return obj


def optimizer_to_device(optimizer, device):
    for state in optimizer.state.values():
        for key, value in state.items():
            if torch.is_tensor(value):
                state[key] = value.to(device)


def pack_rng_state():
    state = {
        "python_random_state": random.getstate(),
        "numpy_random_state": np.random.get_state(),
        "torch_rng_state": torch.get_rng_state().cpu(),
    }
    if torch.cuda.is_available():
        state["torch_cuda_rng_state_all"] = [s.cpu() for s in torch.cuda.get_rng_state_all()]
    return state


def restore_rng_state(state):
    if not state:
        return
    if "python_random_state" in state:
        random.setstate(state["python_random_state"])
    if "numpy_random_state" in state:
        np.random.set_state(state["numpy_random_state"])
    if "torch_rng_state" in state:
        torch.set_rng_state(state["torch_rng_state"])
    if torch.cuda.is_available() and ("torch_cuda_rng_state_all" in state):
        torch.cuda.set_rng_state_all(state["torch_cuda_rng_state_all"])


def build_mode_specs(bundle: QTrafficDataBundle) -> Dict[str, ModeSpec]:
    return {
        "A_traffic_only": ModeSpec("A_traffic_only", "traffic_only", False, "STGCN", "Traffic only + static graph"),
        "B_traffic_time": ModeSpec("B_traffic_time", "traffic_time", False, "STGCN", "Traffic + time + static graph"),
        "C_traffic_time_query": ModeSpec(
            "C_traffic_time_query",
            "traffic_time_weather_query",
            True,
            "CausalSTGCN",
            "Legacy C name; full local query + global weather + dynamic graph",
        ),
        "D_traffic_causal": ModeSpec("D_traffic_causal", "traffic_only", True, "CausalSTGCN", "Traffic only + dynamic graph"),
        "E_traffic_time_causal": ModeSpec(
            "E_traffic_time_causal",
            "traffic_time",
            True,
            "CausalSTGCN",
            "Traffic + time + dynamic graph",
        ),
        "F_traffic_time_query_static": ModeSpec(
            "F_traffic_time_query_static",
            "traffic_time_weather_query",
            False,
            "STGCN",
            "Legacy F name; full local query + global weather + static graph",
        ),
        "G_time_query_static": ModeSpec("G_time_query_static", "traffic_time_query", False, "STGCN", "Traffic + time + query + static graph"),
        "H_time_query_causal": ModeSpec("H_time_query_causal", "traffic_time_query", True, "CausalSTGCN", "Traffic + time + query + dynamic graph"),
        "I_time_weather_static": ModeSpec("I_time_weather_static", "traffic_time_weather", False, "STGCN", "Traffic + time + weather + static graph"),
        "J_time_weather_causal": ModeSpec("J_time_weather_causal", "traffic_time_weather", True, "CausalSTGCN", "Traffic + time + weather + dynamic graph"),
    }


def prepare_paths(cfg: ExperimentConfig, run_name: Optional[str]) -> Dict[str, Path]:
    if run_name:
        root = resolve_cfg_path(Path("./results") / str(run_name), cfg.project_root)
    else:
        root = resolve_cfg_path(cfg.result_dir, cfg.project_root)

    paths = {
        "root": root,
        "run_cache": root / "run_cache",
        "model_ckpt": root / "model_ckpt",
        "epoch_ckpt": root / "epoch_ckpt",
        "state": root / "state",
        "probe": root / "probe",
    }
    for path in paths.values():
        path.mkdir(parents=True, exist_ok=True)
    return paths


def cleanup_setting_epoch_ckpt(epoch_ckpt_dir: Path, rid: str):
    for p in epoch_ckpt_dir.glob(f"{rid}_*.pt"):
        try:
            p.unlink()
        except FileNotFoundError:
            pass


def get_batch_adj_pair(batch_size, a_idx_batch, use_dyn: bool, A_phy_t, A_dyn_t):
    A_phy_batch = A_phy_t.unsqueeze(0).expand(batch_size, -1, -1)
    A_dyn_batch = None
    if use_dyn and (a_idx_batch is not None):
        A_dyn_batch = A_dyn_t[a_idx_batch]
        if A_dyn_batch.dim() == 2:
            A_dyn_batch = A_dyn_batch.unsqueeze(0)
    return A_phy_batch, A_dyn_batch


def get_batch_adj(batch_size, a_idx_batch, lam: float, use_dyn: bool, A_phy_t, A_dyn_t):
    A_phy_batch, A_dyn_batch = get_batch_adj_pair(batch_size, a_idx_batch, use_dyn, A_phy_t, A_dyn_t)
    if A_dyn_batch is None:
        return A_phy_batch
    lam = float(max(0.0, min(1.0, lam)))
    return (1.0 - lam) * A_phy_batch + lam * A_dyn_batch


def forward_with_graphs(model, x, a_idx, use_dyn, lam, A_phy_t, A_dyn_t):
    batch_size = x.shape[0]
    if isinstance(model, CausalSTGCN):
        A_phy_batch, A_dyn_batch = get_batch_adj_pair(batch_size, a_idx, use_dyn, A_phy_t, A_dyn_t)
        return model(x, A_phy_batch, A_dyn_batch, lam=float(lam))
    A_batch = get_batch_adj(batch_size, a_idx, lam, use_dyn, A_phy_t, A_dyn_t)
    return model(x, A_batch), {}


def instantiate_primary_model(cfg: ExperimentConfig, bundle: QTrafficDataBundle, mode_spec: ModeSpec, in_channels: int, H: int):
    schema = bundle.channel_schema[mode_spec.view_key]
    return CausalSTGCN(
        in_channels=in_channels,
        main_channels=int(schema["main_channels"]),
        local_ext_channels=int(schema["local_ext_channels"]),
        global_ext_channels=int(schema["global_ext_channels"]),
        hidden=cfg.hidden,
        blocks=cfg.blocks,
        kt=cfg.kt,
        dilation_base=cfg.dilation_base,
        dropout=cfg.dropout,
        K=cfg.diffusion_K,
        use_backward=cfg.use_backward_diffusion,
        H=H,
    ).to(DEVICE)


def summarize_probe(captured) -> Dict[str, float]:
    if not captured:
        return {}
    aux = captured.get("aux", {})
    out = {}

    local_feat = aux.get("local_node_features")
    if torch.is_tensor(local_feat):
        out["local_node_std"] = float(local_feat.std(dim=1).mean().item())
        out["local_node_abs_mean"] = float(local_feat.abs().mean().item())

    global_feat = aux.get("global_node_features")
    if torch.is_tensor(global_feat):
        out["global_node_std"] = float(global_feat.std(dim=1).mean().item())
        out["global_node_abs_mean"] = float(global_feat.abs().mean().item())

    gate_mean = aux.get("gate_mean")
    if torch.is_tensor(gate_mean):
        out["gate_mean"] = float(gate_mean.item())

    return out


def validate_past_only_alignment(indices, snapshot_end_times, seq_len: int, start_t: int) -> Dict[str, float]:
    if indices is None or len(indices) == 0:
        return {
            "num_samples": 0,
            "max_future_leak": 0.0,
            "max_graph_lag": 0.0,
            "mean_graph_lag": 0.0,
        }

    indices = np.asarray(indices, dtype=np.int64)
    snapshot_end_times = np.asarray(snapshot_end_times, dtype=np.int64)
    input_window_end = int(start_t) + np.arange(len(indices), dtype=np.int64) + int(seq_len) - 1
    graph_times = snapshot_end_times[indices]
    delta = input_window_end - graph_times
    max_future_leak = float(np.max(graph_times - input_window_end))
    if max_future_leak > 0:
        raise RuntimeError(f"Past-only graph alignment violated: max_future_leak={max_future_leak}")
    return {
        "num_samples": int(len(indices)),
        "max_future_leak": max_future_leak,
        "max_graph_lag": float(delta.max()),
        "mean_graph_lag": float(delta.mean()),
    }


def train_one_lambda(
    train_loader,
    val_loader,
    cfg: ExperimentConfig,
    bundle: QTrafficDataBundle,
    mode_spec: ModeSpec,
    in_channels: int,
    H: int,
    use_dyn: bool,
    lam: float,
    model_kind="primary",
    ckpt_path: Optional[Path] = None,
):
    A_phy_t = torch.tensor(bundle.adj_matrix, dtype=torch.float32, device=DEVICE)
    A_dyn_t = torch.tensor(bundle.A_dynamic_tensor, dtype=torch.float32, device=DEVICE)

    if model_kind == "primary":
        model = instantiate_primary_model(cfg, bundle, mode_spec, in_channels, H)
        max_epochs = int(cfg.epochs)
        patience = int(cfg.early_stop_patience)
        label = mode_spec.model_label
    elif model_kind == "dcrnn":
        model = DCRNNBaseline(in_channels=in_channels, hidden=int(cfg.dcrnn_hidden), H=H).to(DEVICE)
        max_epochs = int(cfg.dcrnn_epochs)
        patience = int(cfg.dcrnn_patience)
        label = "DCRNN"
    else:
        raise ValueError(f"unknown model_kind: {model_kind}")

    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    best = inf
    best_state = None
    bad = 0
    start_epoch = 1
    last_epoch = 0
    ckpt_path = None if ckpt_path is None else Path(ckpt_path)

    if bool(cfg.epoch_ckpt_enabled) and (ckpt_path is not None) and ckpt_path.exists():
        try:
            pack = torch.load(ckpt_path, map_location="cpu", weights_only=False)
            saved_kind = str(pack.get("model_kind", ""))
            saved_lam = float(pack.get("lambda", lam))
            if (saved_kind == model_kind) and (abs(saved_lam - float(lam)) < 1e-12):
                model_state = pack.get("model_state")
                if isinstance(model_state, dict) and len(model_state) > 0:
                    model.load_state_dict(model_state)
                opt_state = pack.get("optimizer_state")
                if isinstance(opt_state, dict) and len(opt_state) > 0:
                    opt.load_state_dict(opt_state)
                    optimizer_to_device(opt, DEVICE)
                best = float(pack.get("best_val", best))
                best_state = pack.get("best_state", best_state)
                bad = int(pack.get("bad_count", bad))
                last_epoch = int(pack.get("epoch", 0))
                start_epoch = last_epoch + 1
                if bool(cfg.epoch_ckpt_restore_rng):
                    restore_rng_state(pack.get("rng_state", {}))
                if bool(pack.get("finished", False)):
                    if best_state is not None:
                        model.load_state_dict(best_state)
                    print(f"    [{label}] checkpoint finished | lambda={lam:.3f} | best={best:.4f}")
                    return model, float(best)
                print(f"    [{label}] resume epoch {start_epoch}/{max_epochs} | lambda={lam:.3f} | best={best:.4f}")
        except Exception as e:
            print(f"    [{label}] checkpoint load failed ({ckpt_path.name}): {e}")

    for epoch in range(start_epoch, max_epochs + 1):
        model.train()
        train_loss = 0.0
        for batch in train_loader:
            if len(batch) == 4:
                x, yn, _, m = batch
                a_idx = None
            else:
                x, yn, _, m, a_idx = batch
            x = x.to(DEVICE)
            yn = yn.to(DEVICE)
            m = m.to(DEVICE)
            a_idx = None if a_idx is None else a_idx.to(DEVICE)

            opt.zero_grad(set_to_none=True)
            pred, _ = forward_with_graphs(model, x, a_idx, use_dyn, lam, A_phy_t, A_dyn_t)
            loss = masked_l1_loss(pred, yn, m)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
            train_loss += float(loss.item())
        train_loss /= max(1, len(train_loader))

        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for batch in val_loader:
                if len(batch) == 4:
                    x, yn, _, m = batch
                    a_idx = None
                else:
                    x, yn, _, m, a_idx = batch
                x = x.to(DEVICE)
                yn = yn.to(DEVICE)
                m = m.to(DEVICE)
                a_idx = None if a_idx is None else a_idx.to(DEVICE)
                pred, _ = forward_with_graphs(model, x, a_idx, use_dyn, lam, A_phy_t, A_dyn_t)
                val_loss += float(masked_l1_loss(pred, yn, m).item())
        val_loss /= max(1, len(val_loader))

        if val_loss < best - 1e-6:
            best = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        last_epoch = epoch
        should_save_epoch_ckpt = (
            bool(cfg.epoch_ckpt_enabled)
            and (ckpt_path is not None)
            and (
                (epoch % int(max(1, cfg.epoch_ckpt_save_every)) == 0)
                or (epoch == max_epochs)
                or (bad >= patience)
            )
        )
        if should_save_epoch_ckpt:
            payload = {
                "model_kind": model_kind,
                "lambda": float(lam),
                "use_dyn": bool(use_dyn),
                "H": int(H),
                "in_channels": int(in_channels),
                "epoch": int(epoch),
                "max_epochs": int(max_epochs),
                "patience": int(patience),
                "best_val": float(best),
                "bad_count": int(bad),
                "finished": False,
                "model_state": to_cpu_obj(model.state_dict()),
                "optimizer_state": to_cpu_obj(opt.state_dict()),
                "best_state": to_cpu_obj(best_state) if best_state is not None else None,
                "saved_at": datetime.now().isoformat(),
            }
            if bool(cfg.epoch_ckpt_restore_rng):
                payload["rng_state"] = pack_rng_state()
            try:
                save_torch_atomic(payload, ckpt_path)
            except Exception as e:
                print(f"    [{label}] checkpoint save failed ({ckpt_path.name}): {e}")

        if epoch == 1 or epoch % 10 == 0 or bad >= patience:
            print(f"    [{label}] epoch={epoch:03d} lambda={lam:.3f} train={train_loss:.4f} val={val_loss:.4f}")
        if bad >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    if bool(cfg.epoch_ckpt_enabled) and (ckpt_path is not None):
        payload = {
            "model_kind": model_kind,
            "lambda": float(lam),
            "use_dyn": bool(use_dyn),
            "H": int(H),
            "in_channels": int(in_channels),
            "epoch": int(last_epoch),
            "max_epochs": int(max_epochs),
            "patience": int(patience),
            "best_val": float(best),
            "bad_count": int(bad),
            "finished": True,
            "model_state": to_cpu_obj(model.state_dict()),
            "optimizer_state": to_cpu_obj(opt.state_dict()),
            "best_state": to_cpu_obj(best_state) if best_state is not None else None,
            "saved_at": datetime.now().isoformat(),
        }
        if bool(cfg.epoch_ckpt_restore_rng):
            payload["rng_state"] = pack_rng_state()
        try:
            save_torch_atomic(payload, ckpt_path)
        except Exception as e:
            print(f"    [{label}] final checkpoint save failed ({ckpt_path.name}): {e}")
    return model, float(best)


@torch.no_grad()
def predict(model, loader, use_dyn, lam, bundle: QTrafficDataBundle, capture_aux=False):
    A_phy_t = torch.tensor(bundle.adj_matrix, dtype=torch.float32, device=DEVICE)
    A_dyn_t = torch.tensor(bundle.A_dynamic_tensor, dtype=torch.float32, device=DEVICE)
    model.eval()
    y_pred_norm, y_true_raw, mask_raw = [], [], []
    captured = None
    for batch in loader:
        if len(batch) == 4:
            x, _, yr, m = batch
            a_idx = None
        else:
            x, _, yr, m, a_idx = batch
        x = x.to(DEVICE)
        a_idx = None if a_idx is None else a_idx.to(DEVICE)
        pred, aux = forward_with_graphs(model, x, a_idx, use_dyn, lam, A_phy_t, A_dyn_t)
        y_pred_norm.append(pred.detach().cpu().numpy())
        y_true_raw.append(yr.numpy())
        mask_raw.append(m.numpy())
        if capture_aux and captured is None:
            captured = {
                "x": x.detach().cpu(),
                "a_idx": None if a_idx is None else a_idx.detach().cpu(),
                "aux": to_cpu_obj(aux),
            }
    return np.concatenate(y_pred_norm), np.concatenate(y_true_raw), np.concatenate(mask_raw), captured


def last_value_baseline(X_all_test, H, target_idx):
    last = X_all_test[:, -1, :, target_idx]
    return np.repeat(last[:, None, :], repeats=H, axis=1).astype(np.float32)


def historical_average_baseline(y_train_raw, num_test):
    template = np.mean(y_train_raw, axis=0, keepdims=True).astype(np.float32)
    return np.repeat(template, repeats=int(num_test), axis=0)


def fit_arima_p0(train_raw_t, target_idx=0, p=3):
    series = np.asarray(train_raw_t[:, :, target_idx], dtype=np.float64)
    total_steps, num_nodes = series.shape
    p = int(max(1, min(p, total_steps - 2)))
    betas = np.zeros((num_nodes, p + 1), dtype=np.float64)

    for n in range(num_nodes):
        s = series[:, n]
        X = np.ones((total_steps - p, p + 1), dtype=np.float64)
        for k in range(1, p + 1):
            X[:, k] = s[p - k : total_steps - k]
        y = s[p:]
        beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
        betas[n] = beta

    return betas.astype(np.float32), p


def arima_p0_forecast_from_history(X_all_test, H, target_idx, betas, p):
    hist = np.asarray(X_all_test[:, :, :, target_idx], dtype=np.float32)
    num_samples, seq_len, num_nodes = hist.shape
    p = int(min(p, seq_len))

    state = hist[:, -p:, :].copy()
    out = np.zeros((num_samples, H, num_nodes), dtype=np.float32)
    bias = np.repeat(betas[:, 0][None, :], repeats=num_samples, axis=0).astype(np.float32)
    phi = betas[:, 1:]

    for h in range(H):
        yhat = bias.copy()
        for k in range(1, p + 1):
            yhat = yhat + phi[:, k - 1][None, :] * state[:, -k, :]
        out[:, h, :] = yhat.astype(np.float32)
        state = np.concatenate([state[:, 1:, :], yhat[:, None, :].astype(np.float32)], axis=1)

    return out


def build_snapshot_diagnostics(bundle: QTrafficDataBundle) -> pd.DataFrame:
    A_dyn = np.asarray(bundle.A_dynamic_tensor, dtype=np.float32)
    A_phy = np.asarray(bundle.adj_matrix, dtype=np.float32)
    snapshot_end_times = np.asarray(bundle.snapshot_end_times, dtype=np.int64)
    mask = ~np.eye(A_phy.shape[0], dtype=bool)
    phy_edges = (A_phy > 0)[mask]

    rows = []
    for idx, (graph, end_t) in enumerate(zip(A_dyn, snapshot_end_times)):
        dyn_edges = (graph > 0)[mask]
        intersection = np.logical_and(phy_edges, dyn_edges).sum()
        union = np.logical_or(phy_edges, dyn_edges).sum()
        rows.append(
            {
                "snapshot_idx": int(idx),
                "snapshot_end_time": int(end_t),
                "edge_count": int(dyn_edges.sum()),
                "edge_density": float(dyn_edges.mean()),
                "edge_jaccard": float(intersection / max(1, union)),
                "physical_recall": float(intersection / max(1, phy_edges.sum())),
                "dynamic_precision": float(intersection / max(1, dyn_edges.sum())),
                "mean_edge_weight": float(graph[mask].mean()),
                "max_edge_weight": float(graph[mask].max()),
            }
        )
    return pd.DataFrame(rows)


def run_one_setting(
    cfg: ExperimentConfig,
    bundle: QTrafficDataBundle,
    mode_spec: ModeSpec,
    H: int,
    lambdas: Tuple[float, ...],
    seed: int,
    paths: Dict[str, Path],
    force_rerun=False,
):
    rid = run_id(seed, mode_spec.name, H)
    cache_csv = paths["run_cache"] / f"{rid}.csv"
    model_ckpt = paths["model_ckpt"] / f"{rid}.pt"
    model_meta = paths["model_ckpt"] / f"{rid}.json"
    probe_json = paths["probe"] / f"{rid}.json"

    if (not force_rerun) and cache_csv.exists():
        if bool(cfg.epoch_ckpt_enabled) and (not bool(cfg.epoch_ckpt_keep_finished)):
            cleanup_setting_epoch_ckpt(paths["epoch_ckpt"], rid)
        print(f"[Resume] Use cached result: {cache_csv.name}")
        return pd.read_csv(cache_csv)

    X_mode = bundle.feature_views[mode_spec.view_key]
    x_train_raw_t, x_val_raw_t, x_test_raw_t, tr_t_end, va_t_end = split_timeline(X_mode, cfg)
    x_train, y_train_raw = build_samples(x_train_raw_t, cfg.seq_len, H, cfg.target_feature_idx)
    x_val, y_val_raw = build_samples(x_val_raw_t, cfg.seq_len, H, cfg.target_feature_idx)
    x_test, y_test_raw = build_samples(x_test_raw_t, cfg.seq_len, H, cfg.target_feature_idx)

    if len(x_train) == 0 or len(x_val) == 0 or len(x_test) == 0:
        raise RuntimeError(
            f"Not enough samples after split for H={H}: train/val/test={len(x_train)}/{len(x_val)}/{len(x_test)}"
        )

    mask_train = (y_train_raw != cfg.y_null_val) & np.isfinite(y_train_raw)
    mask_val = (y_val_raw != cfg.y_null_val) & np.isfinite(y_val_raw)
    mask_test = (y_test_raw != cfg.y_null_val) & np.isfinite(y_test_raw)

    scalerX = FeatureScaler().fit(x_train)
    x_train_n = scalerX.transform(x_train)
    x_val_n = scalerX.transform(x_val)
    x_test_n = scalerX.transform(x_test)

    scalerY = TargetScaler().fit(y_train_raw, mask_train)
    y_train_n = scalerY.transform(y_train_raw)
    y_val_n = scalerY.transform(y_val_raw)
    y_test_n = scalerY.transform(y_test_raw)

    a_train = a_val = a_test = None
    align_info = {}
    if mode_spec.use_dyn:
        a_train = build_graph_indices_from_snapshots(len(x_train_n), cfg.seq_len, bundle.snapshot_end_times, start_t=0)
        a_val = build_graph_indices_from_snapshots(len(x_val_n), cfg.seq_len, bundle.snapshot_end_times, start_t=tr_t_end)
        a_test = build_graph_indices_from_snapshots(len(x_test_n), cfg.seq_len, bundle.snapshot_end_times, start_t=va_t_end)

        align_info = {
            "train": validate_past_only_alignment(a_train, bundle.snapshot_end_times, cfg.seq_len, start_t=0),
            "val": validate_past_only_alignment(a_val, bundle.snapshot_end_times, cfg.seq_len, start_t=tr_t_end),
            "test": validate_past_only_alignment(a_test, bundle.snapshot_end_times, cfg.seq_len, start_t=va_t_end),
        }

    train_loader = DataLoader(
        SeqDataset(x_train_n, y_train_n, y_train_raw, mask_train, a_train),
        batch_size=cfg.batch_size,
        shuffle=True,
        drop_last=True,
    )
    val_loader = DataLoader(
        SeqDataset(x_val_n, y_val_n, y_val_raw, mask_val, a_val),
        batch_size=cfg.batch_size,
        shuffle=False,
    )
    test_loader = DataLoader(
        SeqDataset(x_test_n, y_test_n, y_test_raw, mask_test, a_test),
        batch_size=cfg.batch_size,
        shuffle=False,
    )

    rows = []
    best = {"lam": None, "val": float("inf"), "model": None}
    in_channels = x_train_n.shape[-1]

    for lam in lambdas:
        ckpt_path = paths["epoch_ckpt"] / f"{rid}_primary_lam{float(lam):.4f}".replace(".", "p").replace("-", "m")
        ckpt_path = ckpt_path.with_suffix(".pt")
        model, val_loss = train_one_lambda(
            train_loader=train_loader,
            val_loader=val_loader,
            cfg=cfg,
            bundle=bundle,
            mode_spec=mode_spec,
            in_channels=in_channels,
            H=H,
            use_dyn=mode_spec.use_dyn,
            lam=float(lam),
            model_kind="primary",
            ckpt_path=ckpt_path,
        )
        if val_loss < best["val"]:
            best = {"lam": float(lam), "val": float(val_loss), "model": model}

    y_pred_norm, y_true_raw_col, mask_test_col, captured = predict(
        best["model"],
        test_loader,
        use_dyn=mode_spec.use_dyn,
        lam=float(best["lam"]),
        bundle=bundle,
        capture_aux=True,
    )
    y_pred_raw = scalerY.inverse(y_pred_norm)

    for step in cfg.report_steps:
        if step > H:
            continue
        idx = step - 1
        yt = y_true_raw_col[:, idx : idx + 1, :]
        yp = y_pred_raw[:, idx : idx + 1, :]
        m = mask_test_col[:, idx : idx + 1, :]
        met = masked_metrics_np(yt, yp, m, min_denom=cfg.mape_min_denom)
        rows.append(
            {
                "mode": mode_spec.name,
                "view_key": mode_spec.view_key,
                "H_trained": H,
                "report_step": step,
                "lambda": best["lam"],
                "model": mode_spec.model_label,
                "best_val_loss": best["val"],
                "use_dyn": bool(mode_spec.use_dyn),
                **met,
                "seed": int(seed),
            }
        )

    main_ckpt = {
        "run_id": rid,
        "seed": int(seed),
        "mode": mode_spec.name,
        "view_key": mode_spec.view_key,
        "H": int(H),
        "lambda": float(best["lam"]),
        "best_val_loss": float(best["val"]),
        "model": mode_spec.model_label,
        "state_dict": {k: v.detach().cpu() for k, v in best["model"].state_dict().items()},
    }

    if bool(cfg.run_baseline_dcrnn):
        d_use_dyn = bool(cfg.dcrnn_use_dyn and mode_spec.use_dyn)
        d_lambdas = cfg.lambdas_full if d_use_dyn else (0.0,)
        best_d = {"lam": None, "val": float("inf"), "model": None}
        for lam in d_lambdas:
            ckpt_path_d = paths["epoch_ckpt"] / f"{rid}_dcrnn_lam{float(lam):.4f}".replace(".", "p").replace("-", "m")
            ckpt_path_d = ckpt_path_d.with_suffix(".pt")
            model_d, val_loss_d = train_one_lambda(
                train_loader=train_loader,
                val_loader=val_loader,
                cfg=cfg,
                bundle=bundle,
                mode_spec=mode_spec,
                in_channels=in_channels,
                H=H,
                use_dyn=d_use_dyn,
                lam=float(lam),
                model_kind="dcrnn",
                ckpt_path=ckpt_path_d,
            )
            if val_loss_d < best_d["val"]:
                best_d = {"lam": float(lam), "val": float(val_loss_d), "model": model_d}

        y_pred_norm_d, y_true_raw_d, mask_test_d, _ = predict(
            best_d["model"],
            test_loader,
            use_dyn=d_use_dyn,
            lam=float(best_d["lam"]),
            bundle=bundle,
            capture_aux=False,
        )
        y_pred_raw_d = scalerY.inverse(y_pred_norm_d)
        for step in cfg.report_steps:
            if step > H:
                continue
            idx = step - 1
            yt = y_true_raw_d[:, idx : idx + 1, :]
            yp = y_pred_raw_d[:, idx : idx + 1, :]
            m = mask_test_d[:, idx : idx + 1, :]
            met = masked_metrics_np(yt, yp, m, min_denom=cfg.mape_min_denom)
            rows.append(
                {
                    "mode": mode_spec.name,
                    "view_key": mode_spec.view_key,
                    "H_trained": H,
                    "report_step": step,
                    "lambda": best_d["lam"],
                    "model": "DCRNN",
                    "best_val_loss": best_d["val"],
                    "use_dyn": bool(d_use_dyn),
                    **met,
                    "seed": int(seed),
                }
            )

    if bool(cfg.run_baseline_ha):
        ha_pred = historical_average_baseline(y_train_raw, num_test=len(x_test))
        for step in cfg.report_steps:
            if step > H:
                continue
            idx = step - 1
            yt = y_test_raw[:, idx : idx + 1, :]
            yp = ha_pred[:, idx : idx + 1, :]
            m = mask_test[:, idx : idx + 1, :]
            met = masked_metrics_np(yt, yp, m, min_denom=cfg.mape_min_denom)
            rows.append(
                {
                    "mode": mode_spec.name,
                    "view_key": mode_spec.view_key,
                    "H_trained": H,
                    "report_step": step,
                    "lambda": 0.0,
                    "model": "HA",
                    "best_val_loss": float("nan"),
                    "use_dyn": False,
                    **met,
                    "seed": int(seed),
                }
            )

    if bool(cfg.run_baseline_arima):
        betas, p_eff = fit_arima_p0(x_train_raw_t, target_idx=cfg.target_feature_idx, p=cfg.arima_p)
        arima_pred = arima_p0_forecast_from_history(x_test, H, cfg.target_feature_idx, betas, p_eff)
        for step in cfg.report_steps:
            if step > H:
                continue
            idx = step - 1
            yt = y_test_raw[:, idx : idx + 1, :]
            yp = arima_pred[:, idx : idx + 1, :]
            m = mask_test[:, idx : idx + 1, :]
            met = masked_metrics_np(yt, yp, m, min_denom=cfg.mape_min_denom)
            rows.append(
                {
                    "mode": mode_spec.name,
                    "view_key": mode_spec.view_key,
                    "H_trained": H,
                    "report_step": step,
                    "lambda": 0.0,
                    "model": "ARIMA",
                    "best_val_loss": float("nan"),
                    "use_dyn": False,
                    **met,
                    "seed": int(seed),
                }
            )

    lv_pred = last_value_baseline(x_test, H, cfg.target_feature_idx)
    for step in cfg.report_steps:
        if step > H:
            continue
        idx = step - 1
        yt = y_test_raw[:, idx : idx + 1, :]
        yp = lv_pred[:, idx : idx + 1, :]
        m = mask_test[:, idx : idx + 1, :]
        met = masked_metrics_np(yt, yp, m, min_denom=cfg.mape_min_denom)
        rows.append(
            {
                "mode": mode_spec.name,
                "view_key": mode_spec.view_key,
                "H_trained": H,
                "report_step": step,
                "lambda": 0.0,
                "model": "LastValue",
                "best_val_loss": float("nan"),
                "use_dyn": False,
                **met,
                "seed": int(seed),
            }
        )

    df_out = pd.DataFrame(rows)
    save_df_atomic(df_out, cache_csv)
    save_torch_atomic(main_ckpt, model_ckpt)
    save_json_atomic(
        {
            "run_id": rid,
            "seed": int(seed),
            "mode": mode_spec.name,
            "view_key": mode_spec.view_key,
            "H": int(H),
            "lambda": float(best["lam"]),
            "best_val_loss": float(best["val"]),
            "model": mode_spec.model_label,
            "saved_at": datetime.now().isoformat(),
            "graph_alignment": align_info,
            "probe_summary": summarize_probe(captured),
        },
        model_meta,
    )
    save_json_atomic(
        {
            "run_id": rid,
            "mode": mode_spec.name,
            "probe_summary": summarize_probe(captured),
        },
        probe_json,
    )

    if bool(cfg.epoch_ckpt_enabled) and (not bool(cfg.epoch_ckpt_keep_finished)):
        cleanup_setting_epoch_ckpt(paths["epoch_ckpt"], rid)
    print(f"[Save] setting result -> {cache_csv.name}")
    return df_out


PAIR_SPECS = [
    {"label": "time@static(B-A)", "base_mode": "A_traffic_only", "base_model": "STGCN", "target_mode": "B_traffic_time", "target_model": "STGCN"},
    {"label": "dynamic@traffic_only(D-A)", "base_mode": "A_traffic_only", "base_model": "STGCN", "target_mode": "D_traffic_causal", "target_model": "CausalSTGCN"},
    {"label": "dynamic@time(E-B)", "base_mode": "B_traffic_time", "base_model": "STGCN", "target_mode": "E_traffic_time_causal", "target_model": "CausalSTGCN"},
    {"label": "external@static(F-B)", "base_mode": "B_traffic_time", "base_model": "STGCN", "target_mode": "F_traffic_time_query_static", "target_model": "STGCN"},
    {"label": "external@dynamic(C-E)", "base_mode": "E_traffic_time_causal", "base_model": "CausalSTGCN", "target_mode": "C_traffic_time_query", "target_model": "CausalSTGCN"},
    {"label": "dynamic@full(C-F)", "base_mode": "F_traffic_time_query_static", "base_model": "STGCN", "target_mode": "C_traffic_time_query", "target_model": "CausalSTGCN"},
    {"label": "query@static(G-B)", "base_mode": "B_traffic_time", "base_model": "STGCN", "target_mode": "G_time_query_static", "target_model": "STGCN"},
    {"label": "query@dynamic(H-E)", "base_mode": "E_traffic_time_causal", "base_model": "CausalSTGCN", "target_mode": "H_time_query_causal", "target_model": "CausalSTGCN"},
    {"label": "weather@static(I-B)", "base_mode": "B_traffic_time", "base_model": "STGCN", "target_mode": "I_time_weather_static", "target_model": "STGCN"},
    {"label": "weather@dynamic(J-E)", "base_mode": "E_traffic_time_causal", "base_model": "CausalSTGCN", "target_mode": "J_time_weather_causal", "target_model": "CausalSTGCN"},
    {"label": "weather_given_query@static(F-G)", "base_mode": "G_time_query_static", "base_model": "STGCN", "target_mode": "F_traffic_time_query_static", "target_model": "STGCN"},
    {"label": "weather_given_query@dynamic(C-H)", "base_mode": "H_time_query_causal", "base_model": "CausalSTGCN", "target_mode": "C_traffic_time_query", "target_model": "CausalSTGCN"},
    {"label": "query_given_weather@static(F-I)", "base_mode": "I_time_weather_static", "base_model": "STGCN", "target_mode": "F_traffic_time_query_static", "target_model": "STGCN"},
    {"label": "query_given_weather@dynamic(C-J)", "base_mode": "J_time_weather_causal", "base_model": "CausalSTGCN", "target_mode": "C_traffic_time_query", "target_model": "CausalSTGCN"},
]


def collect_contrast_rows_from_summary(df_summary, pair_specs):
    rows = []
    keys = sorted(set((int(r["H_trained"]), int(r["report_step"])) for _, r in df_summary.iterrows()))
    for H, step in keys:
        sub = df_summary[(df_summary["H_trained"] == H) & (df_summary["report_step"] == step)]
        for spec in pair_specs:
            base = sub[(sub["mode"] == spec["base_mode"]) & (sub["model"] == spec["base_model"])]
            target = sub[(sub["mode"] == spec["target_mode"]) & (sub["model"] == spec["target_model"])]
            if len(base) == 0 or len(target) == 0:
                continue
            base = base.iloc[0]
            target = target.iloc[0]
            row = {
                "contrast": spec["label"],
                "H_trained": H,
                "report_step": step,
                "base_mode": spec["base_mode"],
                "target_mode": spec["target_mode"],
                "base_model": spec["base_model"],
                "target_model": spec["target_model"],
            }
            for met in ["MAE", "RMSE", "MAPE"]:
                b = float(base[f"{met}_mean"])
                t = float(target[f"{met}_mean"])
                row[f"{met}_base_mean"] = b
                row[f"{met}_target_mean"] = t
                row[f"{met}_target_gain_pct"] = (b - t) / max(1e-9, abs(b)) * 100.0
            rows.append(row)
    return pd.DataFrame(rows)


def mode_spec_to_json(spec: ModeSpec) -> Dict:
    return {
        "name": spec.name,
        "view_key": spec.view_key,
        "use_dyn": bool(spec.use_dyn),
        "model_label": spec.model_label,
        "desc": spec.desc,
    }


def run_phase2_experiment(
    cfg: ExperimentConfig,
    bundle: QTrafficDataBundle,
    seeds: Iterable[int],
    selected_modes: Optional[Iterable[str]] = None,
    selected_horizons: Optional[Iterable[int]] = None,
    use_fast: bool = False,
    force_rerun: bool = False,
    run_name: Optional[str] = None,
):
    paths = prepare_paths(cfg, run_name)
    mode_specs = build_mode_specs(bundle)
    if selected_modes is not None:
        selected_modes = list(selected_modes)
        mode_specs = {k: v for k, v in mode_specs.items() if k in selected_modes}
    horizons = tuple(selected_horizons) if selected_horizons is not None else tuple(cfg.horizons)

    epochs_orig = cfg.epochs
    lambdas_orig = cfg.lambdas_full
    if use_fast:
        cfg.epochs = min(cfg.epochs, 20)
        cfg.lambdas_full = cfg.lambdas_fast

    resume_csv = paths["state"] / "multiseed_resume.csv"
    resume_json = paths["state"] / "multiseed_resume.json"
    if resume_csv.exists():
        all_runs_df = pd.read_csv(resume_csv)
        done_keys = set(zip(all_runs_df["seed"].astype(int), all_runs_df["mode"].astype(str), all_runs_df["H_trained"].astype(int)))
    else:
        all_runs_df = pd.DataFrame()
        done_keys = set()

    print(f"[Phase2Fix] device={DEVICE}, use_fast={use_fast}")
    print(f"[Phase2Fix] dynamic_graph_source={bundle.dynamic_graph_source}")
    print(f"[Phase2Fix] topology_overlap_proxy={bundle.topology_overlap_proxy}")

    for seed in seeds:
        for mode_name, mode_spec in mode_specs.items():
            for H in horizons:
                key = (int(seed), str(mode_name), int(H))
                if (not force_rerun) and key in done_keys:
                    print(f"[Skip] completed: seed={seed}, mode={mode_name}, H={H}")
                    continue

                lambdas = cfg.lambdas_full if mode_spec.use_dyn else (0.0,)
                df_part = run_one_setting(
                    cfg=cfg,
                    bundle=bundle,
                    mode_spec=mode_spec,
                    H=int(H),
                    lambdas=tuple(float(x) for x in lambdas),
                    seed=int(seed),
                    paths=paths,
                    force_rerun=force_rerun,
                )
                all_runs_df = pd.concat([all_runs_df, df_part], ignore_index=True)
                done_keys.add(key)
                save_df_atomic(all_runs_df, resume_csv)
                save_json_atomic(
                    {
                        "updated_at": datetime.now().isoformat(),
                        "epochs": int(cfg.epochs),
                        "lambdas": list(map(float, cfg.lambdas_full)),
                        "completed_settings": len(done_keys),
                        "completed_seeds": sorted(list({k[0] for k in done_keys})),
                        "selected_modes": sorted(list(mode_specs.keys())),
                        "selected_horizons": list(map(int, horizons)),
                    },
                    resume_json,
                )

    cfg.epochs = epochs_orig
    cfg.lambdas_full = lambdas_orig

    if len(all_runs_df) == 0:
        raise RuntimeError("No results were produced.")

    all_runs_df = all_runs_df.sort_values(["seed", "mode", "H_trained", "model", "report_step"]).reset_index(drop=True)
    summary_df = (
        all_runs_df.groupby(["mode", "view_key", "H_trained", "report_step", "model", "use_dyn"])
        .agg(
            MAE_mean=("MAE", "mean"),
            MAE_std=("MAE", "std"),
            RMSE_mean=("RMSE", "mean"),
            RMSE_std=("RMSE", "std"),
            MAPE_mean=("MAPE", "mean"),
            MAPE_std=("MAPE", "std"),
            keep_mean=("keep%", "mean"),
        )
        .reset_index()
    )
    effect_df = collect_contrast_rows_from_summary(summary_df, PAIR_SPECS)
    graph_diag_df = pd.DataFrame(
        [
            {
                "dynamic_graph_source": bundle.dynamic_graph_source,
                "snapshot_count": int(len(bundle.snapshot_end_times)),
                "snapshot_stride": int(cfg.graph_snapshot_stride),
                "graph_window": int(cfg.graph_history_window),
                **bundle.topology_overlap_proxy,
            }
        ]
    )
    snapshot_diag_df = build_snapshot_diagnostics(bundle)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    runs_path = paths["root"] / f"all_runs_multi_{timestamp}.csv"
    summary_path = paths["root"] / f"summary_multi_{timestamp}.csv"
    effects_path = paths["root"] / f"effects_multi_{timestamp}.csv"
    graph_diag_path = paths["root"] / f"graph_diagnostics_{timestamp}.csv"
    snapshot_diag_path = paths["root"] / f"graph_snapshots_{timestamp}.csv"
    config_path = paths["root"] / f"config_multi_{timestamp}.json"

    save_df_atomic(all_runs_df, runs_path)
    save_df_atomic(summary_df, summary_path)
    save_df_atomic(effect_df, effects_path)
    save_df_atomic(graph_diag_df, graph_diag_path)
    save_df_atomic(snapshot_diag_df, snapshot_diag_path)

    state_meta = {}
    if resume_json.exists():
        with open(resume_json, "r", encoding="utf-8") as f:
            state_meta = json.load(f)

    save_json_atomic(
        {
            "saved_at": datetime.now().isoformat(),
            "device": str(DEVICE),
            "use_fast": bool(use_fast),
            "dynamic_graph_source": bundle.dynamic_graph_source,
            "dynamic_graph_eval_mode": cfg.dynamic_graph_eval_mode,
            "snapshot_count": int(len(bundle.snapshot_end_times)),
            "snapshot_end_times_preview": bundle.snapshot_end_times[: min(10, len(bundle.snapshot_end_times))].tolist(),
            "topology_overlap_proxy": bundle.topology_overlap_proxy,
            "state_meta": state_meta,
            "modes": {name: mode_spec_to_json(spec) for name, spec in mode_specs.items()},
            "config": {
                k: (list(v) if isinstance(v, tuple) else str(v) if isinstance(v, Path) else v)
                for k, v in asdict(cfg).items()
            },
        },
        config_path,
    )

    return {
        "paths": paths,
        "all_runs_df": all_runs_df,
        "summary_df": summary_df,
        "effect_df": effect_df,
        "graph_diag_df": graph_diag_df,
        "snapshot_diag_df": snapshot_diag_df,
        "saved_runs_path": runs_path,
        "saved_summary_path": summary_path,
        "saved_effects_path": effects_path,
        "saved_graph_diag_path": graph_diag_path,
        "saved_snapshot_diag_path": snapshot_diag_path,
        "saved_config_path": config_path,
    }


In [ ]:
# ===== 5. Load bundle =====
cfg = ExperimentConfig()
cfg.qtraffic_dir = "./data/qtraffic"
bundle = load_qtraffic_bundle(cfg)

print("traffic_only:", bundle.traffic_only.shape)
print("traffic_time:", bundle.traffic_time.shape)
print("traffic_time_query:", bundle.traffic_time_query.shape)
print("traffic_time_weather:", bundle.traffic_time_weather.shape)
print("traffic_time_weather_query:", bundle.traffic_time_weather_query.shape)
print("dynamic_graph_source:", bundle.dynamic_graph_source)
print("snapshot_count:", len(bundle.snapshot_end_times))
print("topology_overlap_proxy:", bundle.topology_overlap_proxy)
bundle.channel_schema


In [ ]:
# ===== 6. Run setup =====
RUN_NAME = "qtraffic_phase2_gate_bypass_fix_formal"
SEEDS = [0, 21, 42, 63, 84]
SELECTED_MODES = [
    "A_traffic_only",
    "B_traffic_time",
    "C_traffic_time_query",
    "D_traffic_causal",
    "E_traffic_time_causal",
    "F_traffic_time_query_static",
    "G_time_query_static",
    "H_time_query_causal",
    "I_time_weather_static",
    "J_time_weather_causal",
]
SELECTED_HORIZONS = [12, 36]
USE_FAST = False
FORCE_RERUN = False

cfg = ExperimentConfig()
cfg.qtraffic_dir = "./data/qtraffic"
cfg.result_dir = f"./results/{RUN_NAME}"
cfg.epochs = 80
cfg.batch_size = 64
cfg.run_baseline_ha = True
cfg.run_baseline_arima = True
cfg.run_baseline_dcrnn = True

# If AutoDL memory is tight, try:
# cfg.batch_size = 32
#
# For a quicker test run, try:
# cfg.run_baseline_dcrnn = False

set_seed(SEEDS[0])


In [ ]:
# ===== 7. Execute experiment =====
bundle = load_qtraffic_bundle(cfg)
result = run_phase2_experiment(
    cfg=cfg,
    bundle=bundle,
    seeds=SEEDS,
    selected_modes=SELECTED_MODES,
    selected_horizons=SELECTED_HORIZONS,
    use_fast=USE_FAST,
    force_rerun=FORCE_RERUN,
    run_name=RUN_NAME,
)

summary_df = result["summary_df"]
effect_df = result["effect_df"]
graph_diag_df = result["graph_diag_df"]
snapshot_diag_df = result["snapshot_diag_df"]

print("runs ->", result["saved_runs_path"])
print("summary ->", result["saved_summary_path"])
print("effects ->", result["saved_effects_path"])
print("graph diagnostics ->", result["saved_graph_diag_path"])
print("graph snapshots ->", result["saved_snapshot_diag_path"])
print("config ->", result["saved_config_path"])
summary_df


In [ ]:
# ===== 8. Graph diagnostics =====
graph_diag_df, snapshot_diag_df.head()


In [ ]:
# ===== 9. Structural checks =====
import torch

leak = causal_grad_test()
print("causal_grad_test max future grad =", leak)

schema = bundle.channel_schema["traffic_time_weather_query"]
model = CausalSTGCN(
    in_channels=bundle.traffic_time_weather_query.shape[-1],
    main_channels=int(schema["main_channels"]),
    local_ext_channels=int(schema["local_ext_channels"]),
    global_ext_channels=int(schema["global_ext_channels"]),
    hidden=cfg.hidden,
    blocks=cfg.blocks,
    kt=cfg.kt,
    dilation_base=cfg.dilation_base,
    dropout=0.0,
    K=cfg.diffusion_K,
    use_backward=cfg.use_backward_diffusion,
    H=12,
).to(DEVICE)

sample = torch.tensor(bundle.traffic_time_weather_query[: cfg.seq_len][None], dtype=torch.float32).permute(0, 3, 2, 1).to(DEVICE)
A_phy = torch.tensor(bundle.adj_matrix[None], dtype=torch.float32).to(DEVICE)
A_dyn = torch.tensor(bundle.A_dynamic_tensor[:1], dtype=torch.float32).to(DEVICE)
with torch.no_grad():
    _, aux = model(sample, A_phy, A_dyn, lam=0.5)

local_std = None if aux["local_node_features"] is None else float(aux["local_node_features"].std(dim=1).mean().item())
global_std = None if aux["global_node_features"] is None else float(aux["global_node_features"].std(dim=1).mean().item())
print("local query node std =", local_std)
print("global weather node std =", global_std)

idx_demo = build_graph_indices_from_snapshots(16, cfg.seq_len, bundle.snapshot_end_times, start_t=0)
input_end_demo = cfg.seq_len - 1 + torch.arange(16)
graph_time_demo = torch.tensor(bundle.snapshot_end_times[idx_demo])
print("past_only_ok =", bool(torch.all(graph_time_demo <= input_end_demo)))


In [ ]:
# ===== 10. Reload saved CSVs =====
import pandas as pd

saved_summary = pd.read_csv(result["saved_summary_path"])
saved_effects = pd.read_csv(result["saved_effects_path"])
saved_graph_diag = pd.read_csv(result["saved_graph_diag_path"])
saved_summary, saved_effects, saved_graph_diag
